In [15]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''

mol.basis = 'sto6g'
mol.precision = 1e-10
mol.build()
mf = scf.RHF(mol).density_fit()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Tue May 26 18:31:06 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [16]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.07208475
CCSD Corr = -0.09974974
CCSD(T) Corr = -0.09996469


In [20]:
from pyscf.lno import LNOCCSD, LNOCCSD_T
from pyscf.lno.tools import autofrag_iao

# def test_lno_iao_by_thresh(self):
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
iao_coeff = lo.iao.iao(mol, orbocc)
iao_coeff = lo.orth.vec_lowdin(iao_coeff, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

iao_frag_list = autofrag_iao(moliao)

gamma = 10
threshs = [1e-4]

# refs = [
#     [-0.4054784012,-0.4240686326,-0.4303996712],
#     [-0.4060479828,-0.4245745223,-0.4309965749],
# ]

emp2_iao = np.zeros(len(threshs))
eccsd_iao = np.zeros(len(threshs))
eccsd_t_iao = np.zeros(len(threshs))

for i, thresh in enumerate(threshs):
    mcc = LNOCCSD(mf, iao_coeff, iao_frag_list, frozen=frozen).set(verbose=5)
    mcc.lno_thresh = [thresh*gamma,thresh]
    mcc.kernel()
    emp2_iao[i] = mcc.e_corr_pt2
    eccsd_iao[i] = mcc.e_corr_ccsd
    eccsd_t_iao[i] = mcc.e_corr_ccsd_t


******** <class 'pyscf.lno.lnoccsd.LNOCCSD'> ********
nocc = 8, nmo = 12
frozen orbitals 2
max_memory 4000 MB (current use 365 MB)
nfrag = 6  nlo = 14
frag_lolist = [[0, 1, 2, 3, 4], [5], [6], [7, 8, 9, 10, 11], [12], [13]]
frag_wghtlist = None
lno_type = ['1h', '1h']
lno_thresh = [0.001, 0.0001]
lno_pct_occ = None
lno_norb = None
lo_proj_thresh = 1e-10
lo_proj_thresh_active = 0.1
verbose_imp = 2
_ovL = None
_ovL_to_save = None
force_outcore_ao2mo = False
_match_oldcode = False
_max_las_size_ccsd = 1000
_max_las_size_ccsd_t = 1000
Regularized frag_wghtlist = [1. 1. 1. 1. 1. 1.]
    CPU time for LO and fragment        0.00 sec, wall time      0.00 sec



WARN: Input vhf is not found. Building vhf from SCF MO.

ao2mo est mem= 0.05 MB  avail mem= 3634.17 MB
aux blksize for incore ao2mo: 208/208
    CPU time for Integral xform         0.06 sec, wall time      0.01 sec
LO occ proj: 4 active | 1 standby | 3 orthogonal
LO vir proj: 2 active | 2 standby | 0 orthogonal
    CPU time for Fragment 1 make las      0.51 sec, wall time      0.04 sec
Fragment 1/6  LAS: 5/10 Occ | 4/4 Vir | 9/14 MOs
    impsol:  5 LOs  9/14 MOs  5 occ  4 vir
    CPU time for Fragment 1 imp sol       0.84 sec, wall time      0.06 sec
Fragment 1/6  Sol: E_corr(MP2) = -0.0235776773285954  E_corr(CCSD) = -0.0327354101210497  E_corr(CCSD(T)) = 0
    CPU time for Fragment 1             1.35 sec, wall time      0.10 sec
LO occ proj: 1 active | 0 standby | 7 orthogonal
LO vir proj: 1 active | 0 standby | 3 orthogonal
    CPU time for Fragment 2 make las      0.03 sec, wall time      0.00 sec
Fragment 2/6  LAS: 2/10 Occ | 2/4 Vir | 4/14 MOs
    impsol:  1 LOs  4/14 MOs  2 occ

In [19]:
print(iao_coeff.shape)

(14, 14)


In [12]:
from pyscf.lno import lnoccsd
from afqmc.lno_afqmc import prep, mod_lnoccsd

In [21]:
def run_lnoccsd(
        mf, 
        # options, # qmc parameters
        lo_coeff, 
        frag_lolist,
        nfrozen = 0, 
        thresh = 1e-6, 
        run_frg_list = None,
        atom_group = None, # = mol.elements if frag_lolist is grouped by atoms
        # chol_cut = 1e-5,
        ):

    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=0)
    mlno.lno_thresh = [thresh*10,thresh]
    lno_thresh = mlno.lno_thresh
    nfrag = len(frag_lolist)
    lno_type = ['1h','1h']
    lno_pct_occ = [None, None]
    lno_norb = [[None,None]] * nfrag
    eris = mlno.ao2mo()

    # trial = options["trial"]

    if run_frg_list is None:
        run_frg_list = range(nfrag)

    run_frag_lolist = [frag_lolist[i] for i in run_frg_list]

    print(f'Number of LNO-FRAGMENT: {nfrag}')

    # seeds = random.randint(random.PRNGKey(options["seed"]), shape=(nfrag,), minval=0, maxval=100*nfrag)
    # options["max_error"] = options["max_error"]/np.sqrt(nfrag)
    emp2 = np.zeros(nfrag, dtype='float64')
    ecc  = np.zeros(nfrag, dtype='float64')
    nact = np.zeros(nfrag, dtype='int32')

    # Loop over fragment
    for ifrag, loidx in enumerate(run_frag_lolist):
        print("\n")
        width = 80
        msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
        print(msg.center(width, '='))
        if atom_group is not None:
            print(f"Center Atom {atom_group[ifrag]}")

        orbloc = lo_coeff[:,loidx]
        lno_param = [{'thresh': lno_thresh[i], 'pct_occ': lno_pct_occ[i],
                        'norb': lno_norb[ifrag][i]} for i in [0,1]]

        lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        print(f'number of active occupied orbitals: {nactocc}')
        print(f'number of active virtual orbitals: {nactvir}')

        mo_occ = mlno.mo_occ
        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, len(mo_occ))
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=3)
        mcc._s1e = mlno._s1e
        mcc._h1e = mlno._h1e
        mcc._vhf = mlno._vhf
        if mlno.kwargs_imp is not None:
            mcc = mcc.set(**mlno.kwargs_imp)
        # time0 = time.perf_counter()
        (eorb_mp2, eorb_cc), t1, t2 = \
            mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact) # <<< this is on CPU
        # time1 = time.perf_counter()

        print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
        print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

        prjlo = uocc_loc @ uocc_loc.T.conj()
        # options["seed"] = seeds[ifrag]
        # _, _ = prep_afqmc(mf, lno_coeff, t1, t2, lno_frozen, prjlo, options, chol_cut=chol_cut)
        # run_lnoafqmc(options)
        # os.system(f'mv afqmc.out lnoafqmc.out{run_frg_list[ifrag]+1}')

        emp2[ifrag] = eorb_mp2
        ecc[ifrag]  = eorb_cc
        nact[ifrag] = nactocc + nactvir

    emp2_tot = sum(emp2)
    ecc_tot = sum(ecc)
    orbmax = max(nact)
    print(f'LNO-MP2 Corr Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Corr Energy: {eorb_cc:.8f}')
    print(f'Largest LNO Fragment: {orbmax}')
    return emp2_tot, ecc_tot, orbmax

In [22]:
lo_coeff = iao_coeff
frag_lolist = iao_frag_list
nfrozen = 2
thresh = 3e-5
run_frg_list = [0]
atom_group = moliao.elements

In [26]:
mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=0)
mlno.lno_thresh = [thresh*10,thresh]
lno_thresh = mlno.lno_thresh
nfrag = len(frag_lolist)
lno_type = ['1h','1h']
lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag
eris = mlno.ao2mo()

# trial = options["trial"]

if run_frg_list is None:
    run_frg_list = range(nfrag)

run_frag_lolist = [frag_lolist[i] for i in run_frg_list]

print(f'Number of LNO-FRAGMENT: {nfrag}')

# seeds = random.randint(random.PRNGKey(options["seed"]), shape=(nfrag,), minval=0, maxval=100*nfrag)
# options["max_error"] = options["max_error"]/np.sqrt(nfrag)
emp2 = np.zeros(nfrag, dtype='float64')
ecc  = np.zeros(nfrag, dtype='float64')
nact = np.zeros(nfrag, dtype='int32')

# Loop over fragment
for ifrag, loidx in enumerate(run_frag_lolist):
    print("\n")
    width = 80
    msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
    print(msg.center(width, '='))
    if atom_group is not None:
        print(f"Center Atom {atom_group[ifrag]}")

    orbloc = lo_coeff[:,loidx]
    print(orbloc)
    lno_param = None
    lno_param = [{'thresh': lno_thresh[i], 'pct_occ': lno_pct_occ[i],
                    'norb': lno_norb[ifrag][i]} for i in [0,1]]

    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)
    print(uocc_loc)

    nactocc, nactvir = prep.las_size(mf, lno_frozen)
    print(f'number of active occupied orbitals: {nactocc}')
    print(f'number of active virtual orbitals: {nactvir}')

    mo_occ = mlno.mo_occ
    lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, len(mo_occ))
    mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=3)
    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    # time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 = \
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact) # <<< this is on CPU
    # time1 = time.perf_counter()

    print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    prjlo = uocc_loc @ uocc_loc.T.conj()
    # options["seed"] = seeds[ifrag]
    # _, _ = prep_afqmc(mf, lno_coeff, t1, t2, lno_frozen, prjlo, options, chol_cut=chol_cut)
    # run_lnoafqmc(options)
    # os.system(f'mv afqmc.out lnoafqmc.out{run_frg_list[ifrag]+1}')

    emp2[ifrag] = eorb_mp2
    ecc[ifrag]  = eorb_cc
    nact[ifrag] = nactocc + nactvir

emp2_tot = sum(emp2)
ecc_tot = sum(ecc)
orbmax = max(nact)
print(f'LNO-MP2 Corr Energy: {eorb_mp2:.8f}')
print(f'LNO-CCSD Corr Energy: {eorb_cc:.8f}')
print(f'Largest LNO Fragment: {orbmax}')

Number of LNO-FRAGMENT: 6


=========================== RUNNING LNO-FRAGMENT 1/6 ===========================
Center Atom O
[[ 9.92663514e-01 -2.79225148e-01 -4.46882785e-03 -7.73323486e-03
   4.49551821e-17]
 [ 4.70334545e-02  1.25917066e+00  3.33874933e-02  5.79257980e-02
   4.06067909e-17]
 [ 9.30872853e-03  6.30850214e-02  1.07781005e+00 -1.97348293e-02
   2.04675631e-16]
 [ 1.74930214e-02  1.17754156e-01 -1.93035792e-02  1.05117662e+00
   1.30765417e-16]
 [ 5.70250370e-17  4.99293030e-17  2.05178313e-16  1.33458123e-16
   1.00000066e+00]
 [-4.39230871e-02 -3.04528929e-01  9.83345250e-02 -1.65634732e-01
   1.89305261e-16]
 [-4.41181060e-02 -3.07281529e-01 -1.98886729e-01 -8.08690824e-03
  -3.41138211e-16]
 [-3.96096766e-04 -2.52264386e-03 -1.37715279e-03 -5.61483907e-05
   1.83115060e-16]
 [ 1.70591991e-03  1.10499833e-02  3.91101542e-03 -1.86058191e-04
   1.30453802e-16]
 [-4.96384346e-03 -3.02126493e-02 -2.10443131e-02 -2.19750183e-03
  -8.05906367e-17]
 [ 5.82283327e-04  3.840315

In [28]:
uocc_loc.shape

(5, 5)

In [24]:
def projection_construction(M, thresh, thresh_act=None):
    r''' Given M_{mu,i} = <mu | i> the ovlp between two orthonormal basis, find
    the unitary rotation |j'> = u_ij |i> so that {|j'>} significantly ovlp with
    {|mu>}.

    Three subsets will be returned:
        active  : singular value >  thresh_act
        standby : singular value <= thresh_act but > thresh
        frozen  : singular value <= thresh
    '''
    n, m = M.shape
    e, u = np.linalg.eigh(np.dot(M.T.conj(), M))
    if thresh_act is None: thresh_act = thresh
    assert( thresh_act >= thresh )
    mask_act = abs(e) > thresh_act
    mask_std = np.logical_and(abs(e) > thresh, ~mask_act)
    mask_frz = abs(e) <= thresh
    return u[:,mask_act], u[:,mask_std], u[:,mask_frz]

In [30]:
s1e = mf.get_ovlp()
loidx = iao_frag_list[0]
print(loidx)
orbloc2 = lo_coeff[:,loidx]
# print(orbloc2-orbloc)
uocc_loc2 = orbloc.T.conj() @ s1e @ orbocc
print(uocc_loc2.shape)

[0, 1, 2, 3, 4]
(5, 8)


In [31]:
uocc_loc3, uocc_std3, uocc_orth3 = \
            projection_construction(uocc_loc2, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)

In [35]:
print(uocc_loc3.shape)

(8, 4)
